# Connect a model you own

## Objective

Connect your own Python model and understand what it must return.

Run notebook 2 first. This teaching model predicts one constant value. Replace its calculation with your model's training and prediction code.

Your function receives training data, forecast dates, parameters and a seed. It writes forecasts and fitted model files. AutoForecast checks and scores those forecasts. It keeps negative predictions exactly as supplied. If clipping is part of your model, put that rule in your own recorded model code.


## Step 1: Choose a place for the example files

We will keep data and results outside the library folder. The printed path shows where to find them. Set the environment variable only if you want a different folder.


In [ ]:
# Import the tools used in this step.
import os
from pathlib import Path

# Choose the workspace without changing the library checkout.
work = (
    Path(os.environ.get("AUTOFORECAST_TUTORIAL_DIR", Path.home() / "autoforecast-tutorials"))
    .expanduser()
    .resolve()
)
# Create the folder if this is your first visit.
work.mkdir(parents=True, exist_ok=True)
print(work)

### Understand the output

You should see the full workspace path. Open that folder to find the files created in these lessons. This path is a folder, not a forecast result.


## Step 2: Start the data engine

We will start a small local Spark session with two workers. Spark reads and checks the study data. UTC keeps date and time handling consistent.


In [ ]:
# Import the tools used in this step.
from pyspark.sql import SparkSession

# Use two local workers for this small example.
spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

### Understand the output

Spark may print startup messages. The cell should finish without an exception. No forecasts have been made yet.


Save the function in a real Python file so its source can be retained. The callback must return every supplied series/date pair, exactly once. Use `request.seed` for randomized models.


## Step 3: Write the model function

We will save a small Python function in a file. It predicts one number for every requested product and date, then saves both predictions and model settings.


In [ ]:
# Save real source code so each run can retain a checked copy.
model_source = """import json
import pyarrow as pa
import pyarrow.parquet as pq
from autoforecast.panel import Forecast
from autoforecast.study_tuning import FitPredictResult


def fit_predict(request):
    # Read the constant value from the model profile.
    level = request.model_parameters["level"]
    # Use exactly the products requested by the study.
    products = [row[request.panel.series_col] for row in request.series_mapping.collect()]
    # Put the predictions in the output folder supplied for this fit.
    folder = request.output_dir / "forecasts"
    folder.mkdir()
    # Predict each requested product and future date once.
    rows = [
        {"series_id": product, "date": day, "yhat": level}
        for product in products for day in request.forecast_dates
    ]
    # Save the product/date predictions as Parquet.
    pq.write_table(pa.Table.from_pylist(rows), folder / "part.parquet")

    # Save what is needed to understand or reload this fitted model.
    model_folder = request.output_dir / "model"
    model_folder.mkdir()
    (model_folder / "parameters.json").write_text(json.dumps({"level": level}))
    # Describe the prediction files so AutoForecast can check them.
    forecast = Forecast(
        adapter=request.model_id, path=str(folder), origin=request.origin,
        horizon=len(request.forecast_dates), n_series=len(products),
        disposition={"emitted": len(rows)}, frequency=request.panel.frequency,
        target_estimand=request.panel.target_estimand,
    )
    # Return the forecast description and the model-file format name.
    return FitPredictResult(forecast, "constant-json-v1")
"""
# Create the folder that will hold our saved model code.
model_folder = work / "models"
model_folder.mkdir(exist_ok=True)
_ = (model_folder / "constant.py").write_text(model_source)

### Understand the output

No output is expected. Open `models/constant.py` in the workspace to see the saved function. We have written the model code, but we have not trained it yet.


The profile holds settings. `valid` is the allowed range; `search` is used only when tuning. The source root records model files, not notebook execution history or hidden state. Include your helpers and keep credentials and outputs outside it.


## Step 4: Record the settings and source location

We will save the model profile and tell the registry where the model code lives. This lets a run retain the source files used by the model.


In [ ]:
# Import the tools used in this step.
import yaml

# Save the allowed values and the default constant prediction.
_ = (work / "model.yaml").write_text("""schema_version: 2
model: {id: constant-example, adapter: external}
hyperparameters:
  model.level:
    default: 4.0
    valid: {kind: float, low: -10.0, high: 100.0, log: false}
    search: {kind: float, low: 1.0, high: 10.0, log: false}
""")
# Read the current study settings before updating them.
settings = yaml.safe_load((work / "study.yaml").read_text())
# Retain all model source files from this folder in each run.
settings["runtime"] = {"implementation": {"source_roots": {"model": "models"}}}
# Save the updated settings for the registry to read.
_ = (work / "study.yaml").write_text(yaml.safe_dump(settings))

### Understand the output

No output is expected. `model.yaml` contains the constant-value settings, and `study.yaml` points to the model source folder. These files will be used in the next step.


## Step 5: Run the model

We will load the function from its file and use the study ID from notebook 2. Fixed parameters use the profile defaults without a search. We will save the run path for later lessons.


In [ ]:
# Import the tools used in this step.
import importlib.util

from autoforecast import run_study
from autoforecast.study_registration import StudyRegistry

# Open the registry described by our saved study settings.
registry = StudyRegistry.from_config(work / "study.yaml", spark=spark)

# Load the model from the file we wrote, not from hidden notebook state.
spec = importlib.util.spec_from_file_location("tutorial_constant", model_folder / "constant.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

# Reuse the study ID saved by the registration lesson.
study_id = (work / "study-id.txt").read_text().strip()
# Fit and evaluate the model, keeping all output files.
result = run_study(
    study_id,
    module.fit_predict,
    registry=registry,
    model_config=work / "model.yaml",
    fixed_parameters={},
    tracking="none",
    publish=False,
)
# Remember this run so later notebooks can reopen it.
_ = (work / "latest-run.txt").write_text(str(result.path))
# Show the result folder and a few useful quantities, without internal details.
summary = result.summary()
print("Saved run:", result.path)
print(
    "Products: original =",
    summary["original_series"],
    "retained =",
    summary["retained_series"],
    "excluded =",
    summary["excluded_series"],
)
print("Validation MAE:", result.validation_metrics["mae"])
print("Final-test MAE:", result.testing_metrics["mae"])
print("Table publication:", summary["publication"]["status"])

### Understand the output

You should see a work estimate of two fits, the saved run folder, and product counts of 2 original, 2 retained and 0 excluded. Retained products are the ones included in the experiment. For this unchanged example, validation and final-test MAE are about 3.14 units per product/day. Smaller MAE means smaller average error. `not_requested` means we chose file output with `publish=False`; it is not a training failure. The path after `Saved run:` is the workflow folder to keep.


There were two fits: validation, then refitting on later history for testing. `fixed_parameters={}` uses the defaults without a search. The callback output folder is temporary; keep `result.path`. NaN, infinity, duplicate or missing forecast cells are errors. Already have predictions? `Study.evaluate_forecasts` can check and score them without training.


## Step 6: Close this Spark session

We are finished with Spark in this notebook. Closing it frees resources before the next lesson.


In [ ]:
# Release Spark resources before moving to another notebook.
spark.stop()

### Understand the output

No output is expected. Spark is now closed. Start a fresh kernel when the next notebook asks you to.
